# 06 · Sensibilidad de α y β de la Weighted Focal Loss — documentación y reporte

**Pregunta que responde.** Matriz de riesgos del Prof. Marcos Alania: *«¿Por qué los pesos α = 2,0 y β = 50,0? ¿Fueron elegidos arbitrariamente?»* (sospecha de *cherry-picking*).

**Qué hace este notebook.** Documenta las tres fases de búsqueda, los scripts en el orden de ejecución y **genera tablas y figuras desde el registro, sin reentrenar**.

> ⚠️ **No reentrenar desde aquí.** Las corridas ya se ejecutaron con los scripts de la sección 2.

| Elemento | Valor |
|---|---|
| Configuración base | D' · S4 + hora + ROTI, ventana de 180 min (la mejor de la ablación, notebook 05) |
| Arquitectura | LSTM apilada 64/64, dropout 0.3 |
| Evaluación | Validación (26-ene-2025 a 21-jul-2026, partición cronológica) |
| Criterio de selección | **HSS** (habilidad de alerta), no RMSE en eventos, que premia sobreestimar |
| Pérdida | L = (1/N) · Σ [ e² · w · (\|e\| + ε)^α · p ], con w = β en minutos de evento y p = 2 si se subestima un evento |

## 1. Pasos seguidos

| Fase | Qué se probó | Semillas | Umbral de alerta | Script | Grupo en el registro |
|---|---|---|---|---|---|
| 1 | Grid β ∈ {25, 50, 100, 150, 200, 300} × α ∈ {1.0, 1.5, 2.0} | 1 | Fijo 0.6 | `scripts/correr_alpha_beta.py` | `alpha_beta` |
| — | Se agrega la calibración del umbral por configuración (`barrido_umbral=True` en `correr_variante`) y se amplía el barrido de umbrales a 0.30–0.75 | — | — | `src/s4/pipeline.py`, YAML | — |
| 2 | β ∈ {5, 10, 25, 50, 100} con α = 1.5 | 3 | Fijo 0.6 **y calibrado** | `scripts/correr_alpha_beta_fase2.py` | `beta_fase2` |
| 3 | α ∈ {1.0, 2.0} con β = 10 (α = 1.5 viene de la fase 2) | 3 | Fijo 0.6 y calibrado | `scripts/correr_alpha_fase3.py` | `alpha_fase3` |

**Por qué se calibró el umbral.** Un β alto infla las predicciones: con un umbral fijo de 0.6 genera más alertas y parece cambiar la calidad del modelo cuando en realidad cambia su escala. Para comparar β de forma justa, cada configuración se evalúa con el umbral que maximiza su HSS en validación (columnas `*_opt`). La evaluación definitiva en prueba se hace una sola vez, con el umbral fijado aquí (notebook 09).

## 2. Scripts empleados (en el orden de ejecución)

Ejecutados desde la raíz del proyecto, en segundo plano:

```bash
nohup python scripts/<script>.py > experiments/log_<nombre>.txt 2>&1 &
```

### 2.1 `scripts/correr_alpha_beta.py` — fase 1

```python
cfg = variante(cargar_config("config/ablacion_2025_2026.yaml"), **{"ventanas.lookback": 180})
datos = preparar_datos(cfg)
for beta in [25, 50, 100, 150, 200, 300]:
    for alpha in [1.0, 1.5, 2.0]:
        c = variante(cfg, **{"perdida.beta": float(beta), "perdida.alpha": alpha})
        correr_variante(c, datos, ft.CONJUNTOS["D_solo_ROTI"], f"a{alpha}_b{beta}", "alpha_beta", semillas=[42])
```

### 2.2 `scripts/correr_alpha_beta_fase2.py` — fase 2

```python
cfg = variante(cargar_config("config/ablacion_2025_2026.yaml"), **{"ventanas.lookback": 180, "perdida.alpha": 1.5})
datos = preparar_datos(cfg)
for beta in [5, 10, 25, 50, 100]:
    c = variante(cfg, **{"perdida.beta": float(beta)})
    correr_variante(c, datos, ft.CONJUNTOS["D_solo_ROTI"], f"b{beta}", "beta_fase2", barrido_umbral=True)
```

### 2.3 `scripts/correr_alpha_fase3.py` — fase 3

```python
cfg = variante(cargar_config("config/ablacion_2025_2026.yaml"), **{"ventanas.lookback": 180, "perdida.beta": 10.0})
datos = preparar_datos(cfg)
for alpha in [1.0, 2.0]:
    c = variante(cfg, **{"perdida.alpha": alpha})
    correr_variante(c, datos, ft.CONJUNTOS["D_solo_ROTI"], f"a{alpha}_b10", "alpha_fase3", barrido_umbral=True)
```

## 3. Reporte a partir del registro (no entrena)

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, ruta
from s4.registry import leer, resumir_semillas

cfg = cargar_config("../config/ablacion_2025_2026.yaml")
REG = ruta(cfg, "experimentos") / f"registro_{cfg['nombre']}.csv"
FIG, TAB = ruta(cfg, "reportes") / "figures", ruta(cfg, "reportes") / "tables"
reg = pd.read_csv(REG)
print(reg[reg["grupo"].isin(["alpha_beta", "beta_fase2", "alpha_fase3"])].groupby("grupo").size())

### 3.1 Fase 1: mapas de calor α × β (1 semilla, umbral fijo 0.6)

In [ ]:
f1 = leer(REG, grupo="alpha_beta", particion="val").copy()
f1[["alpha", "beta"]] = f1["modelo"].str.extract(r"a([\d.]+)_b([\d.]+)").astype(float)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (met, tit, cmap) in zip(axes, [("HSS", "HSS", "Blues"), ("Precision", "Precisión", "Greens"), ("POD_inicio", "Inicios anticipados", "Oranges")]):
    m = f1.pivot_table(index="beta", columns="alpha", values=met)
    im = ax.imshow(m.values, cmap=cmap, aspect="auto")
    ax.set_xticks(range(m.shape[1]), [f"α={a:g}" for a in m.columns]); ax.set_yticks(range(m.shape[0]), [f"β={b:g}" for b in m.index])
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]; claro = v > (np.nanmin(m.values) + np.nanmax(m.values)) / 2
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=10, color="white" if claro else "black")
    ax.set_title(tit); plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Fase 1 · al subir β baja la precisión y suben los inicios anticipados (umbral fijo 0.6)")
plt.tight_layout(); plt.savefig(FIG / "fig_alpha_beta_fase1.png", dpi=200); plt.show()
f1.to_csv(TAB / "tabla_alpha_beta_fase1.csv", index=False)

### 3.2 Fase 2: β con umbral fijo frente a umbral calibrado (3 semillas)

In [ ]:
f2 = leer(REG, grupo="beta_fase2", particion="val").copy()
f2["beta"] = f2["modelo"].str.extract(r"b([\d.]+)").astype(float)
t2 = f2.groupby("beta")[["HSS", "HSS_opt", "umbral_opt", "Precision_opt", "POD_inicio_opt"]].agg(["mean", "std"])
t2.columns = [f"{a}_{b}" for a, b in t2.columns]; t2 = t2.reset_index()
display(t2.round(3)); t2.to_csv(TAB / "tabla_beta_fase2.csv", index=False)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.errorbar(t2["beta"], t2["HSS_mean"], yerr=t2["HSS_std"], marker="o", capsize=4, color="#B8BED6", label="HSS con umbral fijo 0.6")
ax1.errorbar(t2["beta"], t2["HSS_opt_mean"], yerr=t2["HSS_opt_std"], marker="o", capsize=4, color="#0000AF", lw=2.5, label="HSS con umbral calibrado")
ax1.axvspan(5, 25, color="#0099FF", alpha=0.12, label="Meseta óptima (β = 5–25)")
ax1.set_xscale("log"); ax1.set_xticks(t2["beta"], [f"{b:g}" for b in t2["beta"]]); ax1.set_xlabel("β (escala logarítmica)"); ax1.set_ylabel("HSS en validación")
ax2 = ax1.twinx(); ax2.plot(t2["beta"], t2["umbral_opt_mean"], "s--", color="#C77C02", label="Umbral óptimo"); ax2.set_ylabel("Umbral de alerta óptimo", color="#C77C02")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax1.legend(h1 + h2, l1 + l2, loc="lower left", fontsize=9)
plt.title("Fase 2 · β cambia sobre todo la escala: al calibrar el umbral, las diferencias se reducen")
plt.tight_layout(); plt.savefig(FIG / "fig_beta_fase2.png", dpi=200); plt.show()

### 3.3 Fase 3: α con β = 10 (3 semillas, umbral calibrado)

In [ ]:
r3 = pd.concat([leer(REG, grupo="alpha_fase3", particion="val"),
                leer(REG, grupo="beta_fase2", particion="val").query("modelo == 'b10'").assign(modelo="a1.5_b10")])
t3 = resumir_semillas(r3, por=("modelo",), metricas=("HSS", "HSS_opt", "umbral_opt", "Precision_opt", "POD_inicio_opt"))
t3.to_csv(TAB / "tabla_alpha_fase3.csv", index=False)
t3.round(3)

## 4. Resultados obtenidos (validación)

### Fase 1 (1 semilla, umbral fijo 0.6) — HSS

| β \ α | 1.0 | 1.5 | 2.0 |
|---|---|---|---|
| 25 | 0.677 | 0.639 | 0.655 |
| 50 | 0.616 | 0.680 | 0.653 |
| 100 | 0.540 | 0.557 | 0.581 |
| 150 | 0.511 | 0.502 | 0.543 |
| 200 | 0.493 | 0.503 | 0.563 |
| 300 | 0.494 | 0.475 | 0.488 |

Al subir β, la precisión baja (de 67 % a 33 %) y los inicios anticipados suben (de 56 % a 87 %).

### Fase 2 (α = 1.5, 3 semillas)

| β | HSS umbral 0.6 | **HSS calibrado** | Umbral óptimo | Precisión* | Inicios* |
|---|---|---|---|---|---|
| 5 | 0.617 ± 0.022 | **0.668 ± 0.008** | 0.50 | 63 % | 65 % |
| 10 | 0.640 ± 0.014 | **0.664 ± 0.003** | 0.55 (las 3 semillas) | 67 % | 58 % |
| 25 | 0.663 ± 0.006 | **0.667 ± 0.010** | 0.58 | 65 % | 62 % |
| 50 | 0.621 ± 0.023 | 0.646 ± 0.012 | 0.65 | 64 % | 59 % |
| 100 | 0.561 ± 0.013 | 0.631 ± 0.023 | 0.68 | 59 % | 61 % |

\* Con el umbral calibrado.

### Fase 3 (β = 10, 3 semillas)

| α | HSS umbral 0.6 | **HSS calibrado** | Umbral óptimo | Precisión* | Inicios* |
|---|---|---|---|---|---|
| 1.0 | 0.643 ± 0.012 | 0.669 ± 0.005 | 0.53 | 66 % ± 7 % | 62 % ± 9 % |
| **1.5** | 0.640 ± 0.014 | **0.664 ± 0.003** | **0.55 (las 3 semillas)** | **67 % ± 2 %** | **58 % ± 3 %** |
| 2.0 | 0.630 ± 0.011 | 0.656 ± 0.014 | 0.53 | 62 % ± 6 % | 63 % ± 5 % |

## 5. Conclusiones

1. **β actúa sobre todo como un cambio de escala.** Un β más alto infla las predicciones y su umbral óptimo sube de 0.50 a 0.68. Con umbral fijo, β parecía controlar un fuerte compromiso entre precisión y anticipación; con el umbral calibrado, las diferencias se reducen a menos de la mitad y los inicios anticipados quedan en 58–65 % para todos los β.
2. **Existe una meseta óptima en β = 5–25** (HSS calibrado ≈ 0.665, empatados dentro de una desviación estándar). β = 50 y β = 100 rinden algo menos (0.646 y 0.631).
3. **α tiene un efecto secundario:** α = 1.0, 1.5 y 2.0 empatan en HSS calibrado (0.656–0.669), dentro de la variación entre corridas.
4. **Regla de decisión:** entre configuraciones empatadas, se elige la más reproducible. **β = 10 y α = 1.5** tienen la menor variación entre semillas (HSS ± 0.003, precisión ± 2 %, inicios ± 3 %) y el **mismo umbral óptimo (0.55) en las tres semillas**, lo que da un umbral de despliegue que no depende de la semilla de entrenamiento.
5. **Por qué antes se usaba β = 100:** la búsqueda anterior seleccionaba por RMSE en eventos, una métrica que premia sobreestimar; con HSS y umbral calibrado, el óptimo baja a la meseta 5–25.

**Configuración elegida:** D' · S4 + hora + ROTI · ventana 180 min · WFL **α = 1.5, β = 10** · umbral de alerta **0.55** (a confirmar en prueba, notebook 09).

**Limitaciones.** Selección y calibración del umbral en la misma partición de validación (optimista pero igual para todas las configuraciones); la misma configuración entrenada en momentos distintos puede variar ≈ 0.03 de HSS por el no-determinismo de la GPU, por lo que no deben interpretarse diferencias menores.